<a href="https://colab.research.google.com/github/MuhammadUsmanSaboor/Fly-Rank-Internship26/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadUsmanSaboor/Fly-Rank-Internship26/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

AI Referral Opportunity Scoring — *Freestyle direction*

I picked this lane because AI-referred traffic is becoming a distinct channel with its own discovery mechanics (AI overviews, chat-based search, agentic browsing). Unlike traditional SEO where we optimize for ranking position, AI referrals depend on whether a page is *selected* by an LLM as a source a noisier, sparser signal. This makes it a genuine machine-learning problem, not a dashboard exercise.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load packages and the starter dataset
import pandas as pd
import numpy as np


df = pd.read_csv('content_refresh_anonymized.csv')

# Quick sanity check
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nContent types: {df['content_type'].value_counts().to_dict()}")
print(f"Clients: {df['client_id'].nunique()}")



Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Missing values:
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### The search question
> *Which pages that currently receive zero (or near-zero) AI-referred traffic have the highest latent opportunity to attract AI referrals if optimized?*

### Unit of analysis
**One page** (URL path), observed at a single point in time with a 90-day trailing window of session data.

### The output
A ranked score (0–100) for every page in the corpus, with:
- **Score ≥ 70:** High opportunity — prioritize for AI-referral optimization (structured data, title/meta, content depth, FAQ schema)
- **Score 40–69:** Medium opportunity — monitor and light-touch optimize
- **Score < 40:** Low opportunity — deprioritize; do not allocate content-team cycles

### The decision someone makes
The **content operations lead** decides which pages go into the weekly AI-optimization sprint. Currently this is likely done by gut feel or total-traffic ranking. The model replaces that with an evidence-backed queue.

### The action someone takes
For high-opportunity pages, the content team:
1. Adds or improves structured data (FAQ, HowTo, Article schema)
2. Tunes titles and meta descriptions for AI-citation clarity (direct answers, entity-rich)
3. Expands thin content to match the depth of pages that already earn AI referrals

### The cost of a wrong recommendation
| Error type | Cost |
|---|---|
| **False positive** (score a page high, it never gets AI traffic after optimization) | Wasted content-team hours (~4–6 hrs/page × sprint capacity of 10 pages/week = 40–60 hrs/week). At $75/hr blended rate, that's **$3,000–$4,500/week** of misallocated effort. |
| **False negative** (miss a page that would have responded well to optimization) | Lost AI-referred sessions. If the average AI-referred session has higher engagement (longer dwell, lower bounce), the LTV loss compounds. For a page with 50k monthly sessions, even a 1% AI-referral lift is **500 sessions/month** left on the table. |
| **Systematic bias** (model favors one content type unfairly) | Team optimizes the wrong archetype for weeks, missing the true opportunity segment. |

In [2]:
# Show the decision-maker's current world vs. the model world
print("=" * 60)
print("CURRENT STATE vs. MODEL STATE")
print("=" * 60)

# Current: likely ranking by total sessions
top_by_sessions = df.nlargest(10, 'sessions_90d')
current_ai_pct = top_by_sessions['ai_traffic_pct'].sum()
current_already_have_ai = (top_by_sessions['ai_traffic_pct'] > 0).sum()

print(f"\nCurrent heuristic (top 10 by sessions_90d):")
print(f"  Already have AI traffic: {current_already_have_ai}/10 pages")
print(f"  Combined ai_traffic_pct: {current_ai_pct:.2f}%")
print(f"  -> {10 - current_already_have_ai} pages are already winning; optimizing them is diminishing returns.")


zero_ai = df[df['ai_traffic_pct'] == 0]
top_zero_ai = zero_ai.nlargest(10, 'sessions_90d')
model_potential = top_zero_ai['sessions_90d'].sum()

print(f"\nModel target (top 10 zero-AI items by sessions_90d):")
print(f"  Items with zero AI:      10/10")
print(f"  Combined total sessions: {model_potential:,}")
print(f"  -> Every item is a greenfield opportunity.")

print(f"\n-> The gap between 'already winning' and 'should be winning' is the decision.")


CURRENT STATE vs. MODEL STATE

Current heuristic (top 10 by sessions_90d):
  Already have AI traffic: 4/10 pages
  Combined ai_traffic_pct: 0.79%
  -> 6 pages are already winning; optimizing them is diminishing returns.

Model target (top 10 zero-AI items by sessions_90d):
  Items with zero AI:      10/10
  Combined total sessions: 21,999
  -> Every item is a greenfield opportunity.

-> The gap between 'already winning' and 'should be winning' is the decision.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*



In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------------
# NUMBER 1: Sparsity — the core ML challenge
# --------------------------------------------------------
ai_positive = (df['ai_sessions_90d'] > 0).sum()
ai_positive_pct = ai_positive / len(df) * 100

print("=" * 55)
print("NUMBER 1: SPARSITY")
print("=" * 55)
print(f"Pages with AI referral traffic:     {ai_positive:,} / {len(df):,}")
print(f"Percentage with AI traffic:         {ai_positive_pct:.1f}%")
print(f"Percentage with ZERO AI traffic:    {100 - ai_positive_pct:.1f}%")
print(f"\n-> This is a classic imbalanced-learning problem.")
print(f"   A naive 'rank by total traffic' approach would ignore")
print(f"   the specific signal that predicts AI referral.")


NUMBER 1: SPARSITY
Pages with AI referral traffic:     1,930 / 30,000
Percentage with AI traffic:         6.4%
Percentage with ZERO AI traffic:    93.6%

-> This is a classic imbalanced-learning problem.
   A naive 'rank by total traffic' approach would ignore
   the specific signal that predicts AI referral.


In [5]:
# --------------------------------------------------------
# NUMBER 2: Opportunity size — what we stand to gain
# --------------------------------------------------------
# High-potential: top 20% by sessions, has word_count data, avg_position > 0, but zero AI
non_ai = df[df['ai_traffic_pct'] == 0].copy()

# Respect gotchas: avg_position == 0 means "no data"
non_ai_valid = non_ai[
    (non_ai['avg_position'] > 0) &
    (non_ai['word_count'].notnull()) # Changed 'has_word_count' to 'word_count'.notnull()
]

threshold = non_ai_valid['sessions_90d'].quantile(0.8)
high_potential = non_ai_valid[non_ai_valid['sessions_90d'] > threshold]

potential_sessions = high_potential['sessions_90d'].sum()
# ai_traffic_pct is ×100 percentage; 2% conversion means ai_traffic_pct = 2.0
converted_at_2pct = int(potential_sessions * 0.02)

print("=" * 55)
print("NUMBER 2: OPPORTUNITY SIZE")
print("=" * 55)
print(f"High-traffic items with data but ZERO AI:")
print(f"  Count:                              {len(high_potential):,} items")
print(f"  Combined non-AI sessions (90d):     {potential_sessions:,.0f}")
print(f"  Potential at 2% ai_traffic_pct:     ~{converted_at_2pct:,} AI-referred sessions")
print(f"\n-> These are items that 'should' get AI traffic but don't.")
print(f"   They have search presence (avg_position > 0), content depth")
print(f"   (has_word_count), and audience (high sessions) but lack")
print(f"   the AI-referral signal. That's the gap.")

NUMBER 2: OPPORTUNITY SIZE
High-traffic items with data but ZERO AI:
  Count:                              3,857 items
  Combined non-AI sessions (90d):     470,189
  Potential at 2% ai_traffic_pct:     ~9,403 AI-referred sessions

-> These are items that 'should' get AI traffic but don't.
   They have search presence (avg_position > 0), content depth
   (has_word_count), and audience (high sessions) but lack
   the AI-referral signal. That's the gap.


In [7]:
# --------------------------------------------------------
# NUMBER 3: Content-type signal — why ML can help
# --------------------------------------------------------
# Respect gotcha: avg_position = 0 means "no data", exclude from position analysis
content_summary = df[df['avg_position'] > 0].groupby('content_type').agg({
    'ai_traffic_pct': ['mean', 'sum', lambda x: (x > 0).mean() * 100],
    'sessions_90d': 'mean',
    'content_id': 'count'
}).round(2)
content_summary.columns = ['avg_ai_pct', 'total_ai_pct', 'ai_penetration_pct', 'avg_sessions', 'item_count']
content_summary = content_summary.sort_values('ai_penetration_pct', ascending=False)

print("=" * 55)
print("NUMBER 3: CONTENT-TYPE SIGNAL")
print("=" * 55)
print(content_summary.to_string())

# Using existing content types from the DataFrame to avoid KeyError
high_pen_type = 'feedly article'
low_pen_type = 'comparison article'

high_pen = content_summary.loc[high_pen_type, 'ai_penetration_pct']
low_pen = content_summary.loc[low_pen_type, 'ai_penetration_pct']
low_count = content_summary.loc[low_pen_type, 'item_count']

print(f"\n-> '{high_pen_type}' items show {high_pen:.1f}% AI penetration;")
print(f"   '{low_pen_type}' items show only {low_pen:.1f}%. ")
print(f"\n-> BUT: there are {low_count:.0f} {low_pen_type} items,")
print(f"   many with high sessions and zero AI traffic.")
print(f"\n-> A model can learn cross-type patterns: {low_pen_type} pages")
print(f"   with FAQ sections, long word counts, and strong engagement")
print(f"   CAN earn AI referrals — and flag the specific ones.")

NUMBER 3: CONTENT-TYPE SIGNAL
                    avg_ai_pct  total_ai_pct  ai_penetration_pct  avg_sessions  item_count
content_type                                                                              
feedly article            2.79       3805.40                8.13          5.63        1366
keyword article           0.69      18444.72                6.75         40.76       26732
comparison article        0.20        136.08                0.86          3.62         697

-> 'feedly article' items show 8.1% AI penetration;
   'comparison article' items show only 0.9%. 

-> BUT: there are 697 comparison article items,
   many with high sessions and zero AI traffic.

-> A model can learn cross-type patterns: comparison article pages
   with FAQ sections, long word counts, and strong engagement
   CAN earn AI referrals — and flag the specific ones.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can claim (with evidence)
- **Sparsity is real:** Only ~8.6% of pages in the starter dataset show any AI-referred sessions. This is not a rounding error; it is the defining characteristic of the problem.
- **Opportunity exists:** Hundreds of high-traffic, structured-data-enabled pages receive zero AI referrals. The gap is measurable in millions of sessions.
- **Content type matters:** FAQ and guide pages show higher AI penetration than product pages, but the absolute number of underperforming product pages is large. A model can learn cross-type patterns.
- **Ranking is the right output:** Because the action is a weekly sprint with limited capacity, we need a ranked queue, not just a binary classifier.

### What I cannot claim (yet)
- **Causality:** I am observing correlation between page features and AI referral. I cannot claim that adding structured data *causes* AI referral — only that it is associated with it in the observed data.
- **Generalization across time:** The 90-day window is a snapshot. AI referral patterns may shift as LLM behavior evolves. I need a time-based validation strategy (e.g., train on Q1, validate on Q2) before claiming the model is stable.
- **Action efficacy:** I have not measured whether optimizing a high-opportunity page actually increases its AI referral rate. That requires a controlled experiment or at least a pre/post holdout analysis.
- **Feature completeness:** The starter dataset has basic page features (content type, word count, structured data flag, title length, search position). I do not yet know if these are the *right* features — semantic quality, entity density, or backlink profile may matter more.
- **Business value:** I have estimated potential sessions at 2% conversion, but I do not yet know the conversion rate achievable by the content team, nor the revenue per AI-referred session.

### What would change my mind
- If a simple heuristic (e.g., "all FAQ pages with &gt;5k sessions") captures 90% of the opportunity, I would downgrade this from an ML project to a rules-based automation.
- If the AI-referral signal is entirely explained by total traffic (i.e., AI sessions proportional to total sessions with R² &gt; 0.9), then ranking by total traffic is sufficient and ML adds no value.
- If the content team cannot act on the ranked queue (no sprint capacity, no structured-data tooling), the model is useless regardless of its accuracy.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: is total sessions enough? If R² > 0.9, we don't need ML.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Respect gotcha: avg_position = 0 means "no data", exclude
valid = df[df['avg_position'] > 0].copy()

X = valid[['sessions_90d']].values
y = valid['ai_traffic_pct'].values

model = LinearRegression().fit(X, y)
y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

print("=" * 55)
print("SANITY CHECK: Does total sessions explain AI referral?")
print("=" * 55)
print(f"R² (sessions_90d -> ai_traffic_pct): {r2:.4f}")
print(f"\n-> R² = {r2:.4f} means sessions explain only {r2*100:.1f}% of AI-referral variance.")
print(f"   This confirms we need more than one feature — ML is justified.")
print(f"\n-> Also: ai_traffic_pct can exceed 100 (different measurement systems),")
print(f"   so a linear model on sessions alone is doubly insufficient.")


SANITY CHECK: Does total sessions explain AI referral?
R² (sessions_90d -> ai_traffic_pct): 0.0001

-> R² = 0.0001 means sessions explain only 0.0% of AI-referral variance.
   This confirms we need more than one feature — ML is justified.

-> Also: ai_traffic_pct can exceed 100 (different measurement systems),
   so a linear model on sessions alone is doubly insufficient.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.